In [ ]:
# Databricks notebook source

# =========================
# SETUP
# =========================
from pyspark.sql.functions import monotonically_increasing_id

catalog       = "cabpluse360_team2"
gold_schema   = "cabpulse360_gold_team2"
silver_schema = "cabpulse360_silver_team2"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {gold_schema}")

dbutils.widgets.text("batch_id", "batch_01")
batch_id = dbutils.widgets.get("batch_id")

print(f"Catalog      : {catalog}")
print(f"Gold schema  : {gold_schema}")
print(f"Silver schema: {silver_schema}")
print(f"Batch ID     : {batch_id}")

# =========================
# IMPORTS
# =========================
from pyspark.sql.functions import col, lit, current_timestamp

# =========================
# DQ FUNCTION
# =========================
results = []

def run_test(test_name, test_query):
    try:
        result_value = spark.sql(test_query).collect()[0]["result"]
        status = "PASS" if result_value == 0 else "FAIL"
        results.append({
            "test_name": test_name,
            "result_value": int(result_value),
            "status": status
        })
        print(f"[{status}] {test_name}: value={result_value}")
    except Exception as e:
        results.append({
            "test_name": test_name,
            "result_value": -1,
            "status": "ERROR"
        })
        print(f"[ERROR] {test_name}: {str(e)}")

# =========================
# RUN ALL 10 DQ TESTS
# =========================
print("\n--- Running DQ Tests ---\n")

run_test("fact_trips_no_null_date_key",
    "SELECT COUNT(*) as result FROM fact_trips WHERE DATE_KEY IS NULL")

run_test("fact_trips_no_null_vendor_sk",
    "SELECT COUNT(*) as result FROM fact_trips WHERE VENDOR_SK IS NULL")

run_test("fact_trips_row_count_matches_silver",
    f"""SELECT ABS((SELECT COUNT(*) FROM fact_trips) -
        (SELECT COUNT(*) FROM {catalog}.{silver_schema}.trips)) as result""")

run_test("fact_trips_no_duplicates",
    """SELECT COUNT(*) as result FROM (
        SELECT DATE_KEY, VENDOR_SK, PICKUP_ZONE_SK, COUNT(*) as cnt
        FROM fact_trips GROUP BY 1,2,3 HAVING cnt > 1)""")

run_test("fact_trips_no_zero_fares",
    "SELECT COUNT(*) as result FROM fact_trips WHERE FARE_AMOUNT = 0 AND TOTAL_AMOUNT > 0")

run_test("fact_trips_no_impossible_distance",
    "SELECT COUNT(*) as result FROM fact_trips WHERE TRIP_DISTANCE > 200")

run_test("fact_trips_no_zero_passengers",
    "SELECT COUNT(*) as result FROM fact_trips WHERE PASSENGER_COUNT = 0")

run_test("fact_trips_no_future_dates",
    "SELECT COUNT(*) as result FROM fact_trips WHERE TRIP_DATE > '2019-12-31'")

run_test("fact_trips_no_negative_fares",
    "SELECT COUNT(*) as result FROM fact_trips WHERE FARE_AMOUNT < 0")

run_test("dim_vendor_scd2_has_current",
    """SELECT COUNT(*) as result FROM (
        SELECT VENDOR_ID FROM dim_vendor
        GROUP BY VENDOR_ID HAVING SUM(IS_CURRENT) = 0)""")

# =========================
# SAVE DQ RESULTS TO LOG TABLE
# =========================
print("\n--- Saving DQ results to log table ---\n")

df_results = spark.createDataFrame(results) \
    .withColumn("batch_id", lit(batch_id)) \
    .withColumn("run_timestamp", current_timestamp())

df_results.write \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog}.{gold_schema}.dq_log")

print(f"DQ results saved to {catalog}.{gold_schema}.dq_log")

# =========================
# DQ SUMMARY
# =========================
pass_count = sum(1 for r in results if r["status"] == "PASS")
fail_count = sum(1 for r in results if r["status"] == "FAIL")

print("")
print("=" * 60)
print(f"DQ SUMMARY — {batch_id}")
print("=" * 60)
print(f"Total: {len(results)} | Passed: {pass_count} | Failed: {fail_count}")

if fail_count > 0:
    print("\nFAILED TESTS:")
    for r in results:
        if r["status"] == "FAIL":
            print(f"  FAIL - {r['test_name']}: {r['result_value']} bad rows")

# =========================
# QUARANTINE — only runs if DQ failed
# =========================
if fail_count > 0:
    print("\n--- DQ Failed — Starting Quarantine Process ---\n")

    # Read silver trips for this batch
    df_silver = spark.table(f"{catalog}.{silver_schema}.trips") \
        .filter(col("_batch_id") == batch_id)

    total_rows = df_silver.count()
    print(f"Total rows in silver for {batch_id}: {total_rows}")

    # Define bad row conditions — matching exactly the DQ tests above
    bad_condition = (
        ((col("FARE_AMOUNT") == 0) & (col("TOTAL_AMOUNT") > 0)) |
        (col("TRIP_DISTANCE") > 200)                            |
        (col("PASSENGER_COUNT") == 0)                           |
        (col("TRIP_DATE") > "2019-12-31")                       |
        (col("FARE_AMOUNT") < 0)
    )

    # Split into bad and good
    df_bad  = df_silver.filter(bad_condition) \
        .withColumn("quarantine_reason",    lit("failed_dq_checks")) \
        .withColumn("quarantine_timestamp", current_timestamp()) \
        .withColumn("batch_id",             lit(batch_id))

    df_good = df_silver.filter(~bad_condition)

    bad_count  = df_bad.count()
    good_count = df_good.count()

    print(f"Bad rows  (quarantined): {bad_count}")
    print(f"Good rows (proceeding) : {good_count}")
    print(f"Quarantine %           : {round(bad_count / total_rows * 100, 2)}%")

    # Save bad rows to quarantine table
    df_bad.write \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(f"{catalog}.{gold_schema}.quarantine_trips")

    print(f"\nBad rows saved to {catalog}.{gold_schema}.quarantine_trips")

    clean_table = f"{catalog}.{silver_schema}.trips_clean_{batch_id}"

    if good_count > 0:
        df_good.write \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(clean_table)

        print(f"Clean data saved: {clean_table}")

    # STOP if all bad
    if good_count == 0:
        raise Exception(f"All rows invalid for {batch_id}")


    print("\n--- Rebuilding Gold using clean data ---\n")

    df_t = spark.table(clean_table)

    df_dv = spark.table(f"{catalog}.{gold_schema}.dim_vendor")
    df_dz = spark.table(f"{catalog}.{gold_schema}.dim_taxi_zone")
    df_dr = spark.table(f"{catalog}.{gold_schema}.dim_rate_code")
    df_dp = spark.table(f"{catalog}.{gold_schema}.dim_payment_type")
    df_tb = spark.table(f"{catalog}.{gold_schema}.dim_time_block")
    df_dd = spark.table(f"{catalog}.{gold_schema}.dim_date")

    df_fact = df_t.alias("t") \
        .join(df_dv.alias("v"),
            (col("t.VENDOR_ID") == col("v.VENDOR_ID")) &
            (col("t.TRIP_DATE") >= col("v.EFFECTIVE_DATE")) &
            (col("t.TRIP_DATE") <= col("v.END_DATE")),
            "left") \
        .join(df_dz.alias("pu"), col("t.PICKUP_LOCATION_ID") == col("pu.LOCATION_ID"), "left") \
        .join(df_dz.alias("do"), col("t.DROPOFF_LOCATION_ID") == col("do.LOCATION_ID"), "left") \
        .join(df_dr.alias("r"), col("t.RATE_CODE_ID") == col("r.RATE_CODE_ID"), "left") \
        .join(df_dp.alias("p"), col("t.PAYMENT_TYPE") == col("p.PAYMENT_TYPE_ID"), "left") \
        .join(df_tb.alias("tb"), col("t.TIME_BLOCK") == col("tb.TIME_BLOCK"), "left") \
        .join(df_dd.alias("d"), col("t.TRIP_DATE") == col("d.DATE_ACTUAL"), "left")

    df_fact_trips = df_fact.select(
        monotonically_increasing_id().alias("TRIP_FACT_SK"),
        col("d.DATE_KEY"),
        col("v.VENDOR_SK"),
        col("pu.ZONE_SK").alias("PICKUP_ZONE_SK"),
        col("do.ZONE_SK").alias("DROPOFF_ZONE_SK"),
        col("r.RATE_CODE_SK"),
        col("p.PAYMENT_TYPE_SK"),
        col("tb.TIME_BLOCK_SK"),
        col("t.TRIP_DATE"),
        col("t.PASSENGER_COUNT"),
        col("t.TRIP_DISTANCE"),
        col("t.FARE_AMOUNT"),
        col("t.EXTRA"),
        col("t.TIP_AMOUNT"),
        col("t.TOLLS_AMOUNT"),
        col("t.TOTAL_AMOUNT")
    )

    df_fact_trips.write \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{catalog}.{gold_schema}.fact_trips")

    print("Gold fact_trips rebuilt successfully with clean data")

else:
    print("\nALL DQ CHECKS PASSED — proceeding to export")

In [ ]:
%sql
SELECT * 
FROM cabpluse360_team2.cabpulse360_gold_team2.dq_log
ORDER BY run_timestamp DESC;

In [ ]:
%sql
SELECT * 
FROM cabpluse360_team2.cabpulse360_gold_team2.quarantine_trips;